# 分箱模块演示 (Binning Module)

本notebook演示hscredit库中分箱模块的全部功能，包含16种分箱方法。

In [1]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os

# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)
print(f"数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")
print(f"\n目标列分布:\n{df['FPD15'].value_counts()}")

/Users/xiaoxi/anaconda3/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据形状: (12448, 85)

列名: ['MOB1', 'MOB2', 'loan_date', 'bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3', 'dxm_v6pro', 'bcf_fpv1', 'apply_for_admission_confidence', 'apply_for_admission_score', 'charging_fail_count_12m', 'charging_fail_count_1m', 'charging_fail_count_24m', 'charging_fail_count_3m', 'charging_fail_count_6m', 'consumer_finance_lender_count_12m', 'consumer_finance_lender_count_24m', 'consumer_finance_loan_confidence', 'consumer_finance_loan_credit_line', 'consumer_finance_loan_credit_line_avg', 'consumer_finance_loan_credit_line_max', 'consumer_finance_loan_lender_count', 'consumer_finance_loan_product_count', 'credit_loan_duration', 'hit_consumer_finance_lender_count', 'hit_lender_count', 'hit_network_loan_lender_count', 'last_performance_days', 'lender_count_12m', 'lender_count_1m', 'lender_count_24m', 'lender_count_3m', 'lender_count_6m', 'lender_inquiry_count', 'lender_inquiry_count_1m', 'lender_inquiry_count_3m', 'lender_inquiry_count_6m', 'loan_amt_between_1k_3k_coun

In [2]:
# 定义目标列和排除列
target_col = 'FPD15'
exclude_cols = ['MOB1', 'MOB2', 'loan_date', 'FPD15', 'SFPD15']

# 获取特征列
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"特征数量: {len(feature_cols)}")
print(f"特征列: {feature_cols[:10]}...")  # 只显示前10个

# 准备数据
X = df[feature_cols]
y = df[target_col]
print(f"\nX形状: {X.shape}, y形状: {y.shape}")

特征数量: 80
特征列: ['bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3', 'dxm_v6pro', 'bcf_fpv1', 'apply_for_admission_confidence', 'apply_for_admission_score', 'charging_fail_count_12m', 'charging_fail_count_1m']...

X形状: (12448, 80), y形状: (12448,)


## 1. 导入分箱模块

使用OptimalBinning统一接口访问所有16种分箱方法。

In [3]:
from hscredit.core.binning import (
    OptimalBinning,
    UniformBinning,
    QuantileBinning,
    TreeBinning,
    ChiMergeBinning,
    BestKSBinning,
    BestIVBinning,
    MDLPBinning,
    ORBinning,
    CartBinning,
    KMeansBinning,
    MonotonicBinning,
    GeneticBinning,
    SmoothBinning,
    KernelDensityBinning,
    BestLiftBinning,
    TargetBadRateBinning
)
print("所有分箱方法已导入成功！")

所有分箱方法已导入成功！


## 2. 选择演示特征

选择数值型特征进行演示。

In [4]:
# 选择数值型特征
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
print(f"数值型特征数量: {len(numeric_features)}")

# 选择一个特征进行演示
demo_feature = numeric_features[0] if numeric_features else feature_cols[0]
print(f"演示特征: {demo_feature}")

X_demo = X[[demo_feature]].copy()
print(f"\n特征统计:\n{X_demo.describe()}")

数值型特征数量: 80
演示特征: bj_qy24

特征统计:
         bj_qy24
count 12448.0000
mean    598.5537
std      67.1235
min    -999.0000
25%     553.0000
50%     599.0000
75%     643.0000
max     850.0000


## 3. 各分箱方法演示

### 3.1 UniformBinning (等宽分箱)

In [5]:
# 等宽分箱
binner_uniform = UniformBinning(max_n_bins=5)
binner_uniform.fit(X_demo, y)

# 转换数据
X_binned = binner_uniform.transform(X_demo, metric='indices')
X_woe = binner_uniform.transform(X_demo, metric='woe')

print("=== UniformBinning 等宽分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")
print(f"\nWOE编码:\n{X_woe.head(10)}")

# 获取分箱表
bin_table = binner_uniform.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== UniformBinning 等宽分箱 ===
分箱索引:
   bj_qy24
0        4
1        4
2        4
3        4
4        4
5        3
6        4
7        4
8        4
9        4

WOE编码:
   bj_qy24
0   0.0000
1   0.0000
2   0.0000
3   0.0000
4   0.0000
5   0.0000
6   0.0000
7   0.0000
8   0.0000
9   0.0000

分箱统计表:
   分箱            分箱标签   样本总数   好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率   分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0  (-inf, -629.2]      1      1     0 0.0001 0.0001 0.0000 0.0000 -20.3818 0.0018 0.0125 0.0000 -0.0001   0.0000 -0.0001       1       0 0.0001
1   3  (110.4, 480.2]    446    400    46 0.0358 0.0344 0.0557 0.1031   0.4812 0.0102 0.0125 1.5543  0.0206   1.5509  0.0205     401      46 0.0212
2   4   (480.2, +inf]  12001  11221   780 0.9641 0.9655 0.9443 0.0650  -0.0222 0.0005 0.0125 0.9795 -0.5509   1.0000  0.0000   11622     826 0.0000


### 3.2 QuantileBinning (等频分箱)

In [6]:
# 等频分箱
binner_quantile = QuantileBinning(max_n_bins=5)
binner_quantile.fit(X_demo, y)

X_binned = binner_quantile.transform(X_demo, metric='indices')

print("=== QuantileBinning 等频分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_quantile.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== QuantileBinning 等频分箱 ===
分箱索引:
   bj_qy24
0        2
1        2
2        3
3        2
4        1
5        0
6        0
7        4
8        0
9        2

分箱统计表:
   分箱            分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0   (-inf, 543.0]  2480  2258   222 0.1992 0.1943 0.2688 0.0895  0.3245 0.0242 0.0763 1.3490  0.0868   1.3490  0.0868    2258     222 0.0745
1   1  (543.0, 581.0]  2489  2296   193 0.2000 0.1976 0.2337 0.0775  0.1678 0.0061 0.0763 1.1686  0.0421   1.2586  0.1718    4554     415 0.1106
2   2  (581.0, 615.0]  2484  2314   170 0.1996 0.1991 0.2058 0.0684  0.0331 0.0002 0.0763 1.0314  0.0078   1.1829  0.2729    6868     585 0.1173
3   3  (615.0, 655.0]  2503  2367   136 0.2011 0.2037 0.1646 0.0543 -0.2127 0.0083 0.0763 0.8188 -0.0456   1.0914  0.3650    9235     721 0.0783
4   4   (655.0, +inf]  2492  2387   105 0.2002 0.2054 0.1271 0.0421 -0.4798 0.0376 0.0763 0.6350 -0.0914   1.00

### 3.3 TreeBinning (决策树分箱)

In [7]:
# 决策树分箱
binner_tree = TreeBinning(max_n_bins=5)
binner_tree.fit(X_demo, y)

X_binned = binner_tree.transform(X_demo, metric='indices')

print("=== TreeBinning 决策树分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_tree.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== TreeBinning 决策树分箱 ===
分箱索引:
   bj_qy24
0        3
1        2
2        3
3        2
4        1
5        0
6        1
7        4
8        1
9        2

分箱统计表:
   分箱            分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0   (-inf, 491.5]   666   590    76 0.0535 0.0508 0.0920 0.1141  0.5947 0.0245 0.0962 1.7197  0.0407   1.7197  0.0407     590      76 0.0412
1   1  (491.5, 571.5]  3657  3351   306 0.2938 0.2883 0.3705 0.0837  0.2506 0.0206 0.0962 1.2610  0.1086   1.3317  0.1765    3941     382 0.1234
2   2  (571.5, 597.5]  1824  1698   126 0.1465 0.1461 0.1525 0.0691  0.0431 0.0003 0.0962 1.0410  0.0070   1.2454  0.2394    5639     508 0.1298
3   3  (597.5, 674.5]  4755  4494   261 0.3820 0.3867 0.3160 0.0549 -0.2019 0.0143 0.0962 0.8272 -0.1068   1.0630  0.4444   10133     769 0.0591
4   4   (674.5, +inf]  1546  1489    57 0.1242 0.1281 0.0690 0.0369 -0.6187 0.0366 0.0962 0.5556 -0.0630   1.0000 

### 3.4 ChiMergeBinning (卡方分箱)

In [8]:
# 卡方分箱
binner_chi = ChiMergeBinning(max_n_bins=5)
binner_chi.fit(X_demo, y)

X_binned = binner_chi.transform(X_demo, metric='indices')

print("=== ChiMergeBinning 卡方分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_chi.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== ChiMergeBinning 卡方分箱 ===
分箱索引:
   bj_qy24
0        0
1        0
2        0
3        0
4        0
5        0
6        0
7        2
8        0
9        0

分箱统计表:
   分箱              分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0    (-inf, 627.96]  8358  7730   628 0.6714 0.6651 0.7603 0.0751  0.1337 0.0127 0.0635 1.1323  0.2704   1.1323  0.2704    7730     628 0.0952
1   1  (627.96, 631.44]   268   245    23 0.0215 0.0211 0.0278 0.0858  0.2783 0.0019 0.0635 1.2933  0.0065   1.1373  0.3100    7975     651 0.1019
2   2  (631.44, 757.72]  3698  3530   168 0.2971 0.3037 0.2034 0.0454 -0.4010 0.0402 0.0635 0.6846 -0.1333   1.0015  0.1493   11505     819 0.0016
3   3  (757.72, 773.92]    56    50     6 0.0045 0.0043 0.0073 0.1071  0.5238 0.0016 0.0635 1.6147  0.0028   1.0043  0.7784   11555     825 0.0046
4   4    (773.92, +inf]    68    67     1 0.0055 0.0058 0.0012 0.0147 -1.5606 0.0071 0.0635 0.2216 -0

### 3.5 BestKSBinning (最优KS分箱)

In [9]:
# 最优KS分箱
binner_bestks = BestKSBinning(max_n_bins=5)
binner_bestks.fit(X_demo, y)

X_binned = binner_bestks.transform(X_demo, metric='indices')

print("=== BestKSBinning 最优KS分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_bestks.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== BestKSBinning 最优KS分箱 ===
分箱索引:
   bj_qy24
0        4
1        3
2        4
3        3
4        3
5        3
6        3
7        4
8        3
9        3

分箱统计表:
   分箱              分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率   分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0    (-inf, 450.48]    23    23     0 0.0018 0.0020 0.0000 0.0000 -23.5173 0.0465 0.1189 0.0000 -0.0019   0.0000 -0.0019      23       0 0.0020
1   1  (450.48, 453.96]    36    35     1 0.0029 0.0030 0.0012 0.0278  -0.9113 0.0016 0.1189 0.4186 -0.0017   0.2554 -0.0035      58       1 0.0038
2   2  (453.96, 457.44]    26    24     2 0.0021 0.0021 0.0024 0.0769   0.1592 0.0001 0.1189 1.1592  0.0003   0.5319 -0.0032      82       3 0.0034
3   3   (457.44, 603.6]  6558  6021   537 0.5268 0.5181 0.6501 0.0819   0.2271 0.0300 0.1189 1.2340  0.2606   1.2250  0.2575    6103     540 0.1286
4   4     (603.6, +inf]  5805  5519   286 0.4663 0.4749 0.3462 0.0493  -0.3159 0.0406 0.1189 0.7

### 3.6 BestIVBinning (最优IV分箱) - 推荐方法

In [10]:
# 最优IV分箱（推荐）
binner_bestiv = BestIVBinning(max_n_bins=5)
binner_bestiv.fit(X_demo, y)

X_binned = binner_bestiv.transform(X_demo, metric='indices')
X_woe = binner_bestiv.transform(X_demo, metric='woe')
X_bins = binner_bestiv.transform(X_demo, metric='bins')

print("=== BestIVBinning 最优IV分箱（推荐）===")
print(f"分箱索引:\n{X_binned.head(10)}")
print(f"\n分箱标签:\n{X_bins.head(10)}")
print(f"\nWOE编码:\n{X_woe.head(10)}")

bin_table = binner_bestiv.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== BestIVBinning 最优IV分箱（推荐）===
分箱索引:
   bj_qy24
0        3
1        2
2        3
3        2
4        2
5        1
6        2
7        4
8        2
9        2

分箱标签:
            bj_qy24
0   (603.6, 687.12]
1   (488.76, 603.6]
2   (603.6, 687.12]
3   (488.76, 603.6]
4   (488.76, 603.6]
5  (453.96, 488.76]
6   (488.76, 603.6]
7    (687.12, +inf]
8   (488.76, 603.6]
9   (488.76, 603.6]

WOE编码:
   bj_qy24
0  -0.2470
1   0.1711
2  -0.2470
3   0.1711
4   0.1711
5   0.7105
6   0.1711
7  -0.6592
8   0.1711
9   0.1711

分箱统计表:
   分箱              分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0    (-inf, 453.96]    59    58     1 0.0047 0.0050 0.0012 0.0169 -1.4164 0.0054 0.1017 0.2554 -0.0035   0.2554 -0.0035      58       1 0.0038
1   1  (453.96, 488.76]   554   484    70 0.0445 0.0416 0.0847 0.1264  0.7105 0.0306 0.1017 1.9042  0.0421   1.7455  0.0386     542      71 0.0393
2   2   (488.76, 603.6]  6030  5561 

### 3.7 MDLPBinning (MDLP分箱 - 默认方法)

In [11]:
# MDLP分箱
binner_mdlp = MDLPBinning(max_n_bins=5)
binner_mdlp.fit(X_demo, y)

X_binned = binner_mdlp.transform(X_demo, metric='indices')

print("=== MDLPBinning MDLP分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_mdlp.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== MDLPBinning MDLP分箱 ===
分箱索引:
   bj_qy24
0        1
1        0
2        1
3        0
4        0
5        0
6        0
7        1
8        0
9        0

分箱统计表:
   分箱           分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0  (-inf, 597.0]  6093  5592   501 0.4895 0.4812 0.6065 0.0822  0.2316 0.0290 0.0637 1.2392  0.2293   1.2392  0.2293    5592     501 0.1254
1   1  (597.0, +inf]  6355  6030   325 0.5105 0.5188 0.3935 0.0511 -0.2766 0.0347 0.0637 0.7707 -0.2392   1.0000  0.0000   11622     826 0.0000


### 3.8 ORBinning (运筹规划分箱)

In [12]:
try:
    # 运筹规划分箱
    binner_or = ORBinning(max_n_bins=5)
    binner_or.fit(X_demo, y)
    
    X_binned = binner_or.transform(X_demo, metric='indices')
    
    print("=== ORBinning 运筹规划分箱 ===")
    print(f"分箱索引:\n{X_binned.head(10)}")
    
    bin_table = binner_or.get_bin_table(demo_feature)
    print(f"\n分箱统计表:\n{bin_table}")
except Exception as e:
    print(f"ORBinning 需要 ortools 库: {e}")

=== ORBinning 运筹规划分箱 ===
分箱索引:
   bj_qy24
0        3
1        2
2        3
3        2
4        2
5        1
6        2
7        4
8        2
9        2

分箱统计表:
   分箱              分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0    (-inf, 453.96]    59    58     1 0.0047 0.0050 0.0012 0.0169 -1.4164 0.0054 0.1012 0.2554 -0.0035   0.2554 -0.0035      58       1 0.0038
1   1  (453.96, 488.76]   554   484    70 0.0445 0.0416 0.0847 0.1264  0.7105 0.0306 0.1012 1.9042  0.0421   1.7455  0.0386     542      71 0.0393
2   2  (488.76, 607.08]  6366  5876   490 0.5114 0.5056 0.5932 0.0770  0.1598 0.0140 0.1012 1.1600  0.1674   1.2114  0.2698    6418     561 0.1269
3   3   (607.08, 690.6]  4436  4207   229 0.3564 0.3620 0.2772 0.0516 -0.2667 0.0226 0.1012 0.7780 -0.1229   1.0430  0.4748   10625     790 0.0422
4   4     (690.6, +inf]  1033   997    36 0.0830 0.0858 0.0436 0.0348 -0.6772 0.0286 0.1012 0.5252 -0.043

### 3.9 CartBinning (CART分箱)

In [13]:
# CART分箱
binner_cart = CartBinning(max_n_bins=5)
binner_cart.fit(X_demo, y)

X_binned = binner_cart.transform(X_demo, metric='indices')

print("=== CartBinning CART分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_cart.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== CartBinning CART分箱 ===
分箱索引:
   bj_qy24
0        3
1        2
2        3
3        2
4        2
5        0
6        2
7        4
8        2
9        2

分箱统计表:
   分箱            分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0   (-inf, 482.5]   487   439    48 0.0391 0.0378 0.0581 0.0986  0.4308 0.0088 0.0983 1.4854  0.0198   1.4854  0.0198     439      48 0.0203
1   1  (482.5, 488.5]   126   103    23 0.0101 0.0089 0.0278 0.1825  1.1448 0.0217 0.0983 2.7509  0.0179   1.7455  0.0386     542      71 0.0393
2   2  (488.5, 597.5]  5534  5097   437 0.4446 0.4386 0.5291 0.0790  0.1876 0.0170 0.0983 1.1900  0.1521   1.2454  0.2394    5639     508 0.1298
3   3  (597.5, 674.5]  4755  4494   261 0.3820 0.3867 0.3160 0.0549 -0.2019 0.0143 0.0983 0.8272 -0.1068   1.0630  0.4444   10133     769 0.0591
4   4   (674.5, +inf]  1546  1489    57 0.1242 0.1281 0.0690 0.0369 -0.6187 0.0366 0.0983 0.5556 -0.0630   1.0000

### 3.10 KMeansBinning (K-Means聚类分箱)

In [14]:
# K-Means聚类分箱
binner_kmeans = KMeansBinning(max_n_bins=5)
binner_kmeans.fit(X_demo, y)

X_binned = binner_kmeans.transform(X_demo, metric='indices')

print("=== KMeansBinning K-Means聚类分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_kmeans.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== KMeansBinning K-Means聚类分箱 ===
分箱索引:
   bj_qy24
0        2
1        2
2        2
3        2
4        1
5        0
6        0
7        4
8        1
9        2

分箱统计表:
   分箱                                    分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0               (-inf, 525.1680610948174]  1709  1552   157 0.1373 0.1335 0.1901 0.0919  0.3530 0.0200 0.0734 1.3844  0.0612   1.3844  0.0612    1552     157 0.0565
1   1  (525.1680610948174, 580.1745637007359]  3260  3002   258 0.2619 0.2583 0.3123 0.0791  0.1900 0.0103 0.0734 1.1927  0.0684   1.2586  0.1718    4554     415 0.1106
2   2   (580.1745637007359, 629.787058350472]  3528  3309   219 0.2834 0.2847 0.2651 0.0621 -0.0713 0.0014 0.0734 0.9355 -0.0255   1.1245  0.2677    7863     634 0.0910
3   3   (629.787058350472, 686.4883177758106]  2795  2643   152 0.2245 0.2274 0.1840 0.0544 -0.2117 0.0092 0.0734 0.8196 -0.0522   1.0490  0.4785   10506  

### 3.11 MonotonicBinning (单调性约束分箱)

In [15]:
# 单调性约束分箱
binner_monotonic = MonotonicBinning(max_n_bins=5, monotonic='auto')
binner_monotonic.fit(X_demo, y)

X_binned = binner_monotonic.transform(X_demo, metric='indices')

print("=== MonotonicBinning 单调性约束分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_monotonic.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== MonotonicBinning 单调性约束分箱 ===
分箱索引:
   bj_qy24
0        3
1        2
2        3
3        2
4        1
5        0
6        1
7        4
8        1
9        2

分箱统计表:
   分箱            分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0   (-inf, 489.0]   613   542    71 0.0492 0.0466 0.0860 0.1158  0.6115 0.0240 0.0956 1.7455  0.0386   1.7455  0.0386     542      71 0.0393
1   1  (489.0, 572.0]  3710  3399   311 0.2980 0.2925 0.3765 0.0838  0.2526 0.0212 0.0956 1.2633  0.1118   1.3317  0.1765    3941     382 0.1234
2   2  (572.0, 606.0]  2498  2332   166 0.2007 0.2007 0.2010 0.0665  0.0016 0.0000 0.0956 1.0015  0.0004   1.2107  0.2555    6273     548 0.1237
3   3  (606.0, 683.0]  4356  4123   233 0.3499 0.3548 0.2821 0.0535 -0.2292 0.0167 0.0956 0.8061 -0.1044   1.0530  0.4664   10396     781 0.0510
4   4   (683.0, +inf]  1271  1226    45 0.1021 0.1055 0.0545 0.0354 -0.6608 0.0337 0.0956 0.5336 -0.0530   

### 3.12 GeneticBinning (遗传算法分箱)

In [16]:
# 遗传算法分箱
binner_genetic = GeneticBinning(max_n_bins=5, random_state=42)
binner_genetic.fit(X_demo, y)

X_binned = binner_genetic.transform(X_demo, metric='indices')

print("=== GeneticBinning 遗传算法分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_genetic.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== GeneticBinning 遗传算法分箱 ===
分箱索引:
   bj_qy24
0        2
1        1
2        2
3        1
4        1
5        0
6        0
7        4
8        0
9        1

分箱统计表:
   分箱            分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0   (-inf, 546.0]  2663  2424   239 0.2139 0.2086 0.2893 0.0897  0.3273 0.0264 0.0840 1.3525  0.0959   1.3525  0.0959    2424     239 0.0808
1   1  (546.0, 603.0]  3920  3624   296 0.3149 0.3118 0.3584 0.0755  0.1391 0.0065 0.0840 1.1380  0.0634   1.2248  0.2523    6048     535 0.1273
2   2  (603.0, 651.0]  3150  2973   177 0.2531 0.2558 0.2143 0.0562 -0.1771 0.0074 0.0840 0.8468 -0.0519   1.1024  0.3672    9021     712 0.0858
3   3  (651.0, 677.0]  1235  1177    58 0.0992 0.1013 0.0702 0.0470 -0.3662 0.0114 0.0840 0.7078 -0.0322   1.0580  0.4298   10198     770 0.0547
4   4   (677.0, +inf]  1480  1424    56 0.1189 0.1225 0.0678 0.0378 -0.5918 0.0324 0.0840 0.5702 -0.0580   1.0

### 3.13 SmoothBinning (平滑分箱)

In [17]:
# 平滑分箱
binner_smooth = SmoothBinning(max_n_bins=5)
binner_smooth.fit(X_demo, y)

X_binned = binner_smooth.transform(X_demo, metric='indices')

print("=== SmoothBinning 平滑分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_smooth.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== SmoothBinning 平滑分箱 ===
分箱索引:
   bj_qy24
0        1
1        0
2        1
3        0
4        0
5        0
6        0
7        1
8        0
9        0

分箱统计表:
   分箱           分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0  (-inf, 599.0]  6218  5707   511 0.4995 0.4911 0.6186 0.0822  0.2310 0.0295 0.0663 1.2385  0.2380   1.2385  0.2380    5707     511 0.1276
1   1  (599.0, +inf]  6230  5915   315 0.5005 0.5089 0.3814 0.0506 -0.2886 0.0368 0.0663 0.7620 -0.2385   1.0000  0.0000   11622     826 0.0000


### 3.14 KernelDensityBinning (核密度分箱)

In [18]:
# 核密度分箱
binner_kernel = KernelDensityBinning(max_n_bins=5)
binner_kernel.fit(X_demo, y)

X_binned = binner_kernel.transform(X_demo, metric='indices')

print("=== KernelDensityBinning 核密度分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_kernel.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== KernelDensityBinning 核密度分箱 ===
分箱索引:
   bj_qy24
0        0
1        0
2        0
3        0
4        0
5        0
6        0
7        0
8        0
9        0

分箱统计表:
   分箱          分箱标签   样本总数   好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值   坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0  (-inf, +inf)  12448  11622   826 1.0000 1.0000 1.0000 0.0664  0.0000 0.0000 0.0000 1.0000 0.0000   1.0000  0.0000   11622     826 0.0000


### 3.15 BestLiftBinning (Best Lift分箱)

In [19]:
# Best Lift分箱
binner_lift = BestLiftBinning(max_n_bins=5)
binner_lift.fit(X_demo, y)

X_binned = binner_lift.transform(X_demo, metric='indices')

print("=== BestLiftBinning Best Lift分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_lift.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== BestLiftBinning Best Lift分箱 ===
分箱索引:
   bj_qy24
0        2
1        1
2        2
3        2
4        1
5        0
6        0
7        3
8        0
9        2

分箱统计表:
   分箱            分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0   (-inf, 563.0]  3706  3375   331 0.2977 0.2904 0.4007 0.0893  0.3220 0.0355 0.0747 1.3460  0.1467   1.3460  0.1467    3375     331 0.1103
1   1  (563.0, 584.5]  1522  1413   109 0.1223 0.1216 0.1320 0.0716  0.0819 0.0009 0.0747 1.0793  0.0110   1.2683  0.1943    4788     440 0.1207
2   2  (584.5, 626.0]  2925  2743   182 0.2350 0.2360 0.2203 0.0622 -0.0687 0.0011 0.0747 0.9377 -0.0191   1.1497  0.2842    7531     622 0.1050
3   3   (626.0, +inf]  4295  4091   204 0.3450 0.3520 0.2470 0.0475 -0.3544 0.0372 0.0747 0.7158 -0.1497   1.0000  0.0000   11622     826 0.0000


### 3.16 TargetBadRateBinning (目标坏样本率分箱)

In [20]:
# 目标坏样本率分箱
binner_target = TargetBadRateBinning(max_n_bins=5, target_bad_rates=[0.02, 0.05, 0.1, 0.15])
binner_target.fit(X_demo, y)

X_binned = binner_target.transform(X_demo, metric='indices')

print("=== TargetBadRateBinning 目标坏样本率分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_target.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

=== TargetBadRateBinning 目标坏样本率分箱 ===
分箱索引:
   bj_qy24
0        3
1        3
2        3
3        3
4        3
5        1
6        3
7        3
8        3
9        3

分箱统计表:
   分箱            分箱标签   样本总数   好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0   (-inf, 461.0]    113    108     5 0.0091 0.0093 0.0061 0.0442 -0.4286 0.0014 0.0349 0.6668 -0.0031   0.6668 -0.0031     108       5 0.0032
1   1  (461.0, 478.0]    248    219    29 0.0199 0.0188 0.0351 0.1169  0.6223 0.0101 0.0349 1.7622  0.0155   1.4194  0.0125     327      34 0.0130
2   2  (478.0, 488.0]    239    204    35 0.0192 0.0176 0.0424 0.1464  0.8813 0.0219 0.0349 2.2069  0.0236   1.7331  0.0371     531      69 0.0378
3   3   (488.0, +inf]  11848  11091   757 0.9518 0.9543 0.9165 0.0639 -0.0405 0.0015 0.0349 0.9629 -0.7331   1.0000  0.0000   11622     826 0.0000


## 4. OptimalBinning 统一接口

使用OptimalBinning通过method参数调用所有分箱方法。

In [21]:
# 定义要演示的方法
methods = ['uniform', 'quantile', 'tree', 'chi', 'best_ks', 'best_iv', 'mdlp', 'cart', 'kmeans']

print("=== OptimalBinning 统一接口演示 ===")

for method in methods:
    try:
        binner = OptimalBinning(method=method, max_n_bins=5)
        binner.fit(X_demo, y)
        bin_table = binner.get_bin_table(demo_feature)
        iv_total = bin_table['指标IV值'].iloc[0] if '指标IV值' in bin_table.columns else bin_table['分档IV值'].sum()
        print(f"{method}: IV={iv_total:.4f}")
    except Exception as e:
        print(f"{method}: Error - {str(e)[:50]}")

=== OptimalBinning 统一接口演示 ===
uniform: IV=0.0125
quantile: IV=0.0763
tree: IV=0.0962
chi: IV=0.0635
best_ks: IV=0.1189
best_iv: IV=0.1017
mdlp: IV=0.0637
cart: IV=0.0983
kmeans: IV=0.0766


## 5. 单调性约束演示

OptimalBinning支持单调性约束。

In [22]:
# 单调性约束分箱
monotonic_options = [False, 'ascending', 'descending', 'auto', 'peak', 'valley']

print("=== 单调性约束演示 ===")

for monotonic in monotonic_options:
    try:
        binner = OptimalBinning(method='best_iv', max_n_bins=5, monotonic=monotonic)
        binner.fit(X_demo, y)
        bin_table = binner.get_bin_table(demo_feature)
        bad_rates = bin_table['坏样本率'].values
        is_monotonic = all(bad_rates[i] <= bad_rates[i+1] or bad_rates[i] >= bad_rates[i+1] 
                         for i in range(len(bad_rates)-1))
        print(f"monotonic={monotonic}: {'单调' if is_monotonic else '非单调'} | 坏账率={bad_rates}")
    except Exception as e:
        print(f"monotonic={monotonic}: Error - {str(e)[:50]}")

=== 单调性约束演示 ===
monotonic=False: 单调 | 坏账率=[0.016949 0.126354 0.077778 0.052598 0.035461]
monotonic=ascending: 单调 | 坏账率=[0.068085 0.035168]
monotonic=descending: 单调 | 坏账率=[0.115824 0.083827 0.066453 0.053489 0.035405]
monotonic=auto: 单调 | 坏账率=[0.016949 0.126354 0.077778 0.052598 0.035461]
monotonic=peak: 单调 | 坏账率=[0.016949 0.126354 0.077778 0.052598 0.035461]
monotonic=valley: 单调 | 坏账率=[0.016949 0.126354 0.077778 0.052598 0.035461]


## 6. 自定义切分点

使用user_splits参数指定自定义切分点。

In [23]:
# 自定义切分点
custom_splits = {demo_feature: [X_demo[demo_feature].quantile(q) for q in [0.2, 0.4, 0.6, 0.8]]}
print(f"自定义切分点: {custom_splits}")

binner_custom = OptimalBinning(user_splits=custom_splits)
binner_custom.fit(X_demo, y)

X_binned = binner_custom.transform(X_demo, metric='indices')

print("\n=== 自定义切分点分箱 ===")
print(f"分箱索引:\n{X_binned.head(10)}")

bin_table = binner_custom.get_bin_table(demo_feature)
print(f"\n分箱统计表:\n{bin_table}")

自定义切分点: {'bj_qy24': [543.0, 581.0, 615.0, 655.0]}

=== 自定义切分点分箱 ===
分箱索引:
   bj_qy24
0        2
1        2
2        3
3        2
4        1
5        0
6        0
7        4
8        0
9        2

分箱统计表:
   分箱            分箱标签  样本总数  好样本数  坏样本数   样本占比  好样本占比  坏样本占比   坏样本率  分档WOE值  分档IV值  指标IV值  LIFT值    坏账改善  累积LIFT值  累积坏账改善  累积好样本数  累积坏样本数  分档KS值
0   0   (-inf, 543.0]  2480  2258   222 0.1992 0.1943 0.2688 0.0895  0.3245 0.0242 0.0763 1.3490  0.0868   1.3490  0.0868    2258     222 0.0745
1   1  (543.0, 581.0]  2489  2296   193 0.2000 0.1976 0.2337 0.0775  0.1678 0.0061 0.0763 1.1686  0.0421   1.2586  0.1718    4554     415 0.1106
2   2  (581.0, 615.0]  2484  2314   170 0.1996 0.1991 0.2058 0.0684  0.0331 0.0002 0.0763 1.0314  0.0078   1.1829  0.2729    6868     585 0.1173
3   3  (615.0, 655.0]  2503  2367   136 0.2011 0.2037 0.1646 0.0543 -0.2127 0.0083 0.0763 0.8188 -0.0456   1.0914  0.3650    9235     721 0.0783
4   4   (655.0, +inf]  2492  2387   105 0.2002 0.2054 0.1271 0.0421 -0.4

## 7. 规则导出和导入

分箱规则可以导出和导入，实现跨模型复用。

In [24]:
# 导出规则
binner = OptimalBinning(method='best_iv', max_n_bins=5)
binner.fit(X_demo, y)

rules = binner.export()
print("=== 导出分箱规则 ===")
print(f"规则: {rules}")

# 保存到文件
import json
rules_file = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/binning_rules.json'
with open(rules_file, 'w') as f:
    json.dump(rules, f, indent=2)
print(f"\n规则已保存到: {rules_file}")

# 导入规则
with open(rules_file, 'r') as f:
    loaded_rules = json.load(f)

binner_loaded = OptimalBinning()
binner_loaded.load(loaded_rules)

X_binned_loaded = binner_loaded.transform(X_demo, metric='woe')
print(f"\n=== 导入规则后分箱 ===")
print(f"分箱索引:\n{X_binned_loaded.head(10)}")

=== 导出分箱规则 ===
规则: {'bj_qy24': [453.96, 488.76, 603.6, 687.12], '_woe_maps_': {'bj_qy24': {0: -1.416383, 1: 0.710471, 2: 0.17113, 3: -0.246989, 4: -0.659157, -1: 0.0, -2: 0.0}}}

规则已保存到: /Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/binning_rules.json

=== 导入规则后分箱 ===
分箱索引:
   bj_qy24
0  -0.2470
1   0.1711
2  -0.2470
3   0.1711
4   0.1711
5   0.7105
6   0.1711
7  -0.6592
8   0.1711
9   0.1711


## 8. 批量分箱

对多个特征同时进行分箱。

In [25]:
# 选择多个特征
multi_features = numeric_features[:5]  # 选择前5个特征
X_multi = X[multi_features].copy()
print(f"批量分箱特征: {multi_features}")

# 使用OptimalBinning批量分箱
binner_multi = OptimalBinning(method='best_iv', max_n_bins=5)
binner_multi.fit(X_multi, y)

# 转换
X_multi_binned = binner_multi.transform(X_multi, metric='indices')
X_multi_woe = binner_multi.transform(X_multi, metric='woe')

print(f"\n=== 批量分箱结果 ===")
print(f"分箱索引:\n{X_multi_binned.head()}")

# 打印各特征分箱表
for feat in multi_features:
    print(f"\n--- {feat} 分箱表 ---")
    print(binner_multi.get_bin_table(feat)[['分箱', '坏样本率', '分档WOE值']].to_string())

批量分箱特征: ['bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3', 'dxm_v6pro']

=== 批量分箱结果 ===
分箱索引:
   bj_qy24  mobtech80  bairong_v1  xiaoniu_c3  dxm_v6pro
0        3          4           4           3         -1
1        2          3           2           2          1
2        3          4           2           1         -1
3        2          2           1           0         -1
4        2          4           1           0         -1

--- bj_qy24 分箱表 ---
   分箱   坏样本率  分档WOE值
0   0 0.0169 -1.4164
1   1 0.1264  0.7105
2   2 0.0778  0.1711
3   3 0.0526 -0.2470
4   4 0.0355 -0.6592

--- mobtech80 分箱表 ---
   分箱   坏样本率  分档WOE值
0   0 0.0956  0.3968
1   1 0.0659 -0.0068
2   2 0.0941  0.3794
3   3 0.0650 -0.0223
4   4 0.0483 -0.3368

--- bairong_v1 分箱表 ---
   分箱   坏样本率  分档WOE值
0   0 0.1260  0.7070
1   1 0.0726  0.0968
2   2 0.0476 -0.3527
3   3 0.0256 -0.9953
4   4 0.0062 -2.4290

--- xiaoniu_c3 分箱表 ---
   分箱   坏样本率  分档WOE值
0   0 0.1045  0.4961
1   1 0.0745  0.1247
2   2 0.0580 -0.1436
3   3 0.

## 9. 分箱方法对比

对比不同分箱方法的IV值。

In [26]:
# 特征IV对比
all_methods = ['uniform', 'quantile', 'tree', 'chi', 'best_ks', 'best_iv', 'mdlp', 'cart']

comparison_results = []
for feat in numeric_features[:5]:  # 对前5个特征
    X_feat = X[[feat]].copy()
    for method in all_methods:
        try:
            binner = OptimalBinning(method=method, max_n_bins=5)
            binner.fit(X_feat, y)
            bin_table = binner.get_bin_table(feat)
            if '指标IV值' in bin_table.columns:
                iv = bin_table['指标IV值'].iloc[0]
            else:
                iv = bin_table['分档IV值'].sum()
            comparison_results.append({'特征': feat, '方法': method, 'IV': iv})
        except Exception as e:
            pass

comparison_df = pd.DataFrame(comparison_results)
pivot_df = comparison_df.pivot(index='特征', columns='方法', values='IV')
print("=== 各方法IV值对比 ===")
print(pivot_df.round(4).to_string())

=== 各方法IV值对比 ===
方法          best_iv  best_ks   cart    chi   mdlp  quantile   tree  uniform
特征                                                                         
bairong_v1   0.3238   0.2035 0.2603 0.5339 0.2864    0.2604 0.3066   0.2630
bj_qy24      0.1017   0.1189 0.0983 0.0635 0.0637    0.0763 0.0962   0.0125
dxm_v6pro    0.3784   0.4071 0.3276 0.3696 0.3392    0.3117 0.3860   0.9963
mobtech80    0.0997   0.0883 0.0992 0.0810 0.0660    0.0903 0.0996   0.0000
xiaoniu_c3   0.1866   0.1217 0.1826 0.1721 0.1455    0.1455 0.1826   0.0374


## 10. 保存分箱结果

将分箱结果保存到Excel文件。

In [27]:
# 保存分箱结果
from hscredit.report import ExcelWriter, dataframe2excel

output_file = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/01_binning_results.xlsx'

writer = ExcelWriter()

# 保存分箱规则
rules_df = pd.DataFrame([{'特征': k, '切分点': str(v)} for k, v in binner_multi.export_rules().items()])
dataframe2excel(rules_df, writer, sheet_name='分箱规则')

# 保存各特征分箱表
for feat in multi_features:
    bin_table = binner_multi.get_bin_table(feat)
    dataframe2excel(bin_table, writer, sheet_name=f'{feat[:20]}_分箱表')

writer.save(output_file)
print(f"分箱结果已保存到: {output_file}")

分箱结果已保存到: /Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/01_binning_results.xlsx


## 总结

本notebook演示了hscredit库中分箱模块的全部功能：

1. **16种分箱方法**：uniform, quantile, tree, chi, best_ks, best_iv, mdlp, or_tools, cart, kmeans, monotonic, genetic, smooth, kernel_density, best_lift, target_bad_rate

2. **OptimalBinning统一接口**：通过method参数调用所有分箱方法

3. **单调性约束**：支持ascending, descending, auto, peak, valley

4. **自定义切分点**：通过user_splits参数

5. **规则导出/导入**：实现跨模型复用

6. **批量分箱**：对多个特征同时分箱

7. **方法对比**：对比不同方法的IV值